### Config stuff

In [8]:
import random
import pyspark
from pyspark.sql import SparkSession, functions
import ConnectionConfig as cc
from pyspark.sql.functions import *
cc.setupEnvironment()
cc.listEnvironment()

ALLUSERSPROFILE: C:\ProgramData
APPDATA: C:\Users\lzkol\AppData\Roaming
COMMONPROGRAMFILES: C:\Program Files\Common Files
COMMONPROGRAMFILES(X86): C:\Program Files (x86)\Common Files
COMMONPROGRAMW6432: C:\Program Files\Common Files
COMPUTERNAME: LIZA
COMSPEC: C:\WINDOWS\system32\cmd.exe
DRIVERDATA: C:\Windows\System32\Drivers\DriverData
EFC_33840_1262719628: 1
EFC_33840_1592913036: 1
EFC_33840_2283032206: 1
EFC_33840_2775293581: 1
EFC_33840_3789132940: 1
FPS_BROWSER_APP_PROFILE_STRING: Internet Explorer
FPS_BROWSER_USER_PROFILE_STRING: Default
GOPATH: C:\Users\lzkol\go
HOMEDRIVE: C:
HOMEPATH: \Users\lzkol
IPY_INTERRUPT_EVENT: 2112
JAVA_HOME: C:\Program Files\Java\jdk-11\
JPY_INTERRUPT_EVENT: 2112
JPY_PARENT_PID: 1852
JPY_SESSION_NAME: S2_02_DIM_USER.ipynb
LANG: en_US.UTF-8
LANGUAGE: 
LC_ALL: en_US.UTF-8
LOCALAPPDATA: C:\Users\lzkol\AppData\Local
LOGONSERVER: \\LIZA
NUMBER_OF_PROCESSORS: 16
ONEDRIVE: C:\Users\lzkol\OneDrive
OS: Windows_NT
PATH: C:\Users\lzkol\PycharmProjects\sparkdelta

In [9]:
spark = cc.startLocalCluster("dimUser")
spark.getActiveSession()



```




In [10]:
#EXTRACT
cc.set_connectionProfile("velodb")

df_operational_sales_rep = spark.read \
    .format("jdbc") \
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "velo_users") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "userid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 20) \
    .load()

#TRANSFORM
df_operational_sales_rep.createOrReplaceTempView("velo_users")
df_dim_users = spark.sql("select userid, street, number, zipcode, city, country_code, uuid() as user_SK, to_timestamp('1999-01-01','yyyy-MM-dd') as scd_start, to_timestamp('2100-12-12','yyyy-MM-dd') as scd_end, md5(concat(street, number, zipcode, city, country_code)) as md5, True as current from velo_users")

df_dim_users.printSchema()
df_dim_users.show()
#LOAD
(df_dim_users.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("dimUser"))

root
 |-- userid: integer (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- user_SK: string (nullable = false)
 |-- scd_start: timestamp (nullable = true)
 |-- scd_end: timestamp (nullable = true)
 |-- md5: string (nullable = true)
 |-- current: boolean (nullable = false)

+------+--------------------+--------+-------+--------------------+------------+--------------------+-------------------+-------------------+--------------------+-------+
|userid|              street|  number|zipcode|                city|country_code|             user_SK|          scd_start|            scd_end|                 md5|current|
+------+--------------------+--------+-------+--------------------+------------+--------------------+-------------------+-------------------+--------------------+-------+
|     1|         Somméstraat|    156 |   2060

* The function lit() is used when you want a fixed column value for every row. In this case scd_start, scd_end and current.
* The function md5() performs a md5-hash function on the preferred columns. This is needed to detect scd2 changes. When one of the included columns changes, the md5-hash will change. Include all SCD2 columns in the md5-hash function.